# Advanced Gradient Boosting Models

This notebook is training and evaluating advanced gradient boosting algorithms (XGBoost & LightGBM) for employee attrition prediction, comparing them against the Random Forest baseline.

**Goals:**
- Training XGBoost and LightGBM classifiers
- Comparing performance against Random Forest baseline (98.3% accuracy)
- Analyzing feature importance consistency
- Selecting optimal thresholds for each model
- Generating a comprehensive comparison report

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
from pathlib import Path
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, precision_recall_curve, confusion_matrix,
    classification_report
)
from fairlearn.metrics import (
    MetricFrame,
    demographic_parity_ratio,
    selection_rate
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import warnings
warnings.filterwarnings('ignore')

# Loading project configuration
import sys
sys.path.append('..')
from config import HR_DATA_PATH, MODELS_DIR

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

In [ ]:
# Loading data
df = pd.read_csv(HR_DATA_PATH)

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
df.head()

## 2. Preprocessing & Feature Engineering

In [ ]:
df = df.drop('employee_id', axis=1)
print(f"Shape after dropping employee_id: {df.shape}")

# Encoding categorical variables 
salary_categories = [['low', 'medium', 'high']]
salary_encoder = OrdinalEncoder(categories=salary_categories, dtype=int)

dept_encoder = OneHotEncoder(sparse_output=False, drop='first', dtype=int)

# Encode salary
df['salary_encoded'] = salary_encoder.fit_transform(df[['salary']]).flatten()

# OneHot encode department
dept_encoded = dept_encoder.fit_transform(df[['department']])
dept_columns = [f'Dept_{cat}' for cat in dept_encoder.categories_[0][1:]]

for i, col in enumerate(dept_columns):
    df[col] = dept_encoded[:, i]

print("\nEncoding Mappings:")
print(f"Salary (Ordinal): {{0: 'low', 1: 'medium', 2: 'high'}}")
print(f"department (OneHot, baseline=accounting dropped): {dept_columns}")

In [ ]:
# Preparing features and target

# Get all OneHot department columns
dept_columns = [c for c in df.columns if c.startswith('Dept_')]

feature_cols = [
    'satisfaction_level', 
    'last_evaluation', 
    'num_projects',
    'avg_monthly_hours', 
    'years_at_company', 
    'had_work_accident',
    'promotion_last_5_years', 
    'salary_encoded'
] + dept_columns

X = df[feature_cols]
y = df['attrition']

# Calculating class imbalance ratio
neg_count, pos_count = np.bincount(y)
scale_pos_weight = neg_count / pos_count

print(f"Feature matrix shape: {X.shape}")
print(f"Class distribution: {np.bincount(y)}")
print(f"Attrition rate: {y.mean():.2%}")
print(f"Scale position weight: {scale_pos_weight:.2f}")

In [ ]:
# Splitting data with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train/Test Split:")
print(f"Training: {X_train.shape[0]} samples")
print(f"Test: {X_test.shape[0]} samples")
print(f"\nAttrition rate - Train: {y_train.mean():.2%}, Test: {y_test.mean():.2%}")

## 3. Training Gradient Boosting Models

In [ ]:
print("Training Random Forest (Baseline)...")
start_time = time.time()

rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

rf_train_time = time.time() - start_time
rf_pred = rf.predict(X_test)
rf_proba = rf.predict_proba(X_test)[:, 1]

print(f"Training time: {rf_train_time:.2f}s")
print(f"Accuracy: {accuracy_score(y_test, rf_pred):.4f}")
print(f"ROC AUC: {roc_auc_score(y_test, rf_proba):.4f}")

In [ ]:
print("Training XGBoost...")
start_time = time.time()

xgb = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss',
    n_jobs=-1
)
xgb.fit(X_train, y_train)

xgb_train_time = time.time() - start_time
xgb_pred = xgb.predict(X_test)
xgb_proba = xgb.predict_proba(X_test)[:, 1]

print(f"Training time: {xgb_train_time:.2f}s")
print(f"Accuracy: {accuracy_score(y_test, xgb_pred):.4f}")
print(f"ROC AUC: {roc_auc_score(y_test, xgb_proba):.4f}")

In [ ]:
print("Training LightGBM...")
start_time = time.time()

lgb = LGBMClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    class_weight='balanced',
    random_state=42,
    verbose=-1,
    n_jobs=-1
)
lgb.fit(X_train, y_train)

lgb_train_time = time.time() - start_time
lgb_pred = lgb.predict(X_test)
lgb_proba = lgb.predict_proba(X_test)[:, 1]

print(f"Training time: {lgb_train_time:.2f}s")
print(f"Accuracy: {accuracy_score(y_test, lgb_pred):.4f}")
print(f"ROC AUC: {roc_auc_score(y_test, lgb_proba):.4f}")

## 4. Model Comparison

In [ ]:
def get_metrics(y_true, y_pred, y_proba):
    """Getting metrics for model evaluation."""
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred),
        'recall': recall_score(y_true, y_pred),
        'f1': f1_score(y_true, y_pred),
        'roc_auc': roc_auc_score(y_true, y_proba)
    }

comparison = pd.DataFrame({
    'Random Forest': {**get_metrics(y_test, rf_pred, rf_proba), 'train_time': rf_train_time},
    'XGBoost': {**get_metrics(y_test, xgb_pred, xgb_proba), 'train_time': xgb_train_time},
    'LightGBM': {**get_metrics(y_test, lgb_pred, lgb_proba), 'train_time': lgb_train_time}
}).T

print("Model Comparison Summary:")
comparison.style.format({
    'accuracy': '{:.4f}',
    'precision': '{:.4f}',
    'recall': '{:.4f}',
    'f1': '{:.4f}',
    'roc_auc': '{:.4f}',
    'train_time': '{:.2f}s'
}).background_gradient(cmap='YlGnBu', subset=['accuracy', 'roc_auc', 'f1'])

### ROC Curve Comparison

In [ ]:
plt.figure(figsize=(10, 8))

models = [
    ('Random Forest', rf_proba, 'blue'),
    ('XGBoost', xgb_proba, 'green'),
    ('LightGBM', lgb_proba, 'orange')
]

for name, proba, color in models:
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc:.4f})', linewidth=2, color=color)

plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier', linewidth=1)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Cross-Validation Scores

In [ ]:
print("Running 5-fold cross-validation...")

cv_results = {}
for name, model in [('Random Forest', rf), ('XGBoost', xgb), ('LightGBM', lgb)]:
    scores = cross_val_score(model, X, y, cv=5, scoring='roc_auc', n_jobs=-1)
    cv_results[name] = scores
    print(f"{name}: {scores.mean():.4f} (+/- {scores.std() * 2:.4f})")

# Plotting CV scores
plt.figure(figsize=(10, 6))
positions = np.arange(len(cv_results))
means = [cv_results[name].mean() for name in cv_results]
stds = [cv_results[name].std() for name in cv_results]
names = list(cv_results.keys())

plt.bar(positions, means, yerr=stds, alpha=0.7, capsize=5, color=['blue', 'green', 'orange'])
plt.xticks(positions, names)
plt.ylabel('ROC AUC Score')
plt.title('5-Fold Cross-Validation Comparison')
plt.grid(True, alpha=0.3, axis='y')
plt.ylim([0.95, 1.0])
plt.tight_layout()
plt.show()

## 5. Optimal Threshold Analysis

In [ ]:
def find_optimal_threshold(y_true, y_proba):
    """Finding threshold that maximizes F1 score."""
    precision, recall, thresholds = precision_recall_curve(y_true, y_proba)
    f1_scores = 2 * (precision * recall) / (precision + recall + 1e-10)
    optimal_idx = np.argmax(f1_scores)
    return thresholds[optimal_idx], f1_scores[optimal_idx]

optimal_thresholds = {}
for name, proba in [('Random Forest', rf_proba), ('XGBoost', xgb_proba), ('LightGBM', lgb_proba)]:
    thresh, f1 = find_optimal_threshold(y_test, proba)
    optimal_thresholds[name] = thresh
    print(f"{name}: Optimal threshold = {thresh:.4f}, F1 = {f1:.4f}")

## 6. Feature Importance Comparison

In [ ]:
# Extracting feature importance from all models
rf_importance = pd.DataFrame({
    'feature': feature_cols,
    'Random Forest': rf.feature_importances_
})

xgb_importance = pd.DataFrame({
    'feature': feature_cols,
    'XGBoost': xgb.feature_importances_
})

lgb_importance = pd.DataFrame({
    'feature': feature_cols,
    'LightGBM': lgb.feature_importances_
})

# Merging all importances
feature_importance = rf_importance.merge(xgb_importance, on='feature').merge(lgb_importance, on='feature')

# Normalizing to percentages
for col in ['Random Forest', 'XGBoost', 'LightGBM']:
    feature_importance[f'{col}_pct'] = feature_importance[col] / feature_importance[col].sum() * 100

print("Feature Importance Comparison (Top 5 by Random Forest):")
top_features = feature_importance.nlargest(5, 'Random Forest')
top_features.style.format({
    'Random Forest': '{:.4f}',
    'XGBoost': '{:.4f}',
    'LightGBM': '{:.4f}',
    'Random Forest_pct': '{:.1f}',
    'XGBoost_pct': '{:.1f}',
    'LightGBM_pct': '{:.1f}'
})

In [ ]:
# Plotting feature importance comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for idx, model_name in enumerate(['Random Forest', 'XGBoost', 'LightGBM']):
    importance_col = f'{model_name}_pct'
    sorted_features = feature_importance.sort_values(importance_col, ascending=True)
    
    axes[idx].barh(sorted_features['feature'], sorted_features[importance_col], color='steelblue')
    axes[idx].set_xlabel('Importance (%)')
    axes[idx].set_title(f'{model_name} Feature Importance')
    axes[idx].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

## 7. Confusion Matrix Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes = axes.ravel()

predictions = [('Random Forest', rf_pred), ('XGBoost', xgb_pred), ('LightGBM', lgb_pred)]

for idx, (name, pred) in enumerate(predictions):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                xticklabels=['Stay', 'Leave'], yticklabels=['Stay', 'Leave'])
    axes[idx].set_title(name)
    axes[idx].set_ylabel('True Label')
    axes[idx].set_xlabel('Predicted Label')

plt.tight_layout()
plt.show()

## 8. Fairness Analysis Across Models

In [ ]:
# Reconstructing sensitive features

# Mapping encoded salary back to original categories
salary_map = {0: 'low', 1: 'medium', 2: 'high'}
salary_sensitive = X_test['salary_encoded'].map(salary_map)

# Mapping OneHot department back to original category
dept_columns = [c for c in X_test.columns if c.startswith('Dept_')]
dept_sensitive = (
      X_test[dept_columns]
      .idxmax(axis=1)
      .str.replace('Dept_', '')
      .replace('', 'accounting')  # When all zeros, idxmax returns first empty
  )

dept_sensitive = dept_sensitive.fillna('accounting')

In [ ]:
# Computing fairness metrics for all models
metrics_dict = {
    'selection_rate': selection_rate,
    'accuracy': accuracy_score,
    'f1': f1_score
}

fairness_results = {}

for name, model, proba in [
    ('Random Forest', rf, rf_proba),
    ('XGBoost', xgb, xgb_proba),
    ('LightGBM', lgb, lgb_proba)
]:
    pred = (proba >= optimal_thresholds[name]).astype(int)
    
    # Computing demographic parity for salary
    salary_mf = MetricFrame(
        metrics=metrics_dict,
        y_true=y_test.values,
        y_pred=pred,
        sensitive_features=salary_sensitive
    )
    salary_dp_ratio = demographic_parity_ratio(
        y_true=y_test.values,
        y_pred=pred,
        sensitive_features=salary_sensitive
    )
    
    # Computing demographic parity for department
    dept_mf = MetricFrame(
        metrics=metrics_dict,
        y_true=y_test.values,
        y_pred=pred,
        sensitive_features=dept_sensitive
    )
    dept_dp_ratio = demographic_parity_ratio(
        y_true=y_test.values,
        y_pred=pred,
        sensitive_features=dept_sensitive
    )
    
    fairness_results[name] = {
        'salary_dp_ratio': salary_dp_ratio,
        'department_dp_ratio': dept_dp_ratio
    }

# Visualizing fairness comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Salary demographic parity
models = list(fairness_results.keys())
salary_ratios = [fairness_results[m]['salary_dp_ratio'] for m in models]
colors = ['#2ecc71' if r >= 0.8 else '#e74c3c' for r in salary_ratios]
axes[0].bar(models, salary_ratios, color=colors, alpha=0.7)
axes[0].axhline(y=0.8, color='black', linestyle='--', label='EEOC 80% Threshold')
axes[0].set_title('Demographic Parity Ratio (Salary)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Ratio', fontsize=10)
axes[0].set_ylim(0, 1.1)
axes[0].legend()
for i, v in enumerate(salary_ratios):
    axes[0].text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold')

# department demographic parity
dept_ratios = [fairness_results[m]['department_dp_ratio'] for m in models]
colors = ['#2ecc71' if r >= 0.8 else '#e74c3c' for r in dept_ratios]
axes[1].bar(models, dept_ratios, color=colors, alpha=0.7)
axes[1].axhline(y=0.8, color='black', linestyle='--', label='EEOC 80% Threshold')
axes[1].set_title('Demographic Parity Ratio (department)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Ratio', fontsize=10)
axes[1].set_ylim(0, 1.1)
axes[1].legend()
for i, v in enumerate(dept_ratios):
    axes[1].text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 9. Model Performance Summary

In [ ]:
print("=" * 70)
print("FINAL MODEL COMPARISON REPORT")
print("=" * 70)

# Baseline metrics
baseline_acc = comparison.loc['Random Forest', 'accuracy']
baseline_auc = comparison.loc['Random Forest', 'roc_auc']

print(f"\nBaseline (Random Forest): {baseline_acc:.4f} accuracy, {baseline_auc:.4f} ROC AUC")

for model in ['XGBoost', 'LightGBM']:
    acc = comparison.loc[model, 'accuracy']
    auc = comparison.loc[model, 'roc_auc']
    acc_delta = (acc - baseline_acc) * 100
    auc_delta = auc - baseline_auc
    
    status = "Meets or exceeds baseline" if acc >= baseline_acc and auc >= baseline_auc else "~ Below baseline"
    
    print(f"\n{model}:")
    print(f"  Accuracy: {acc:.4f} ({acc_delta:+.2f}%)")
    print(f"  ROC AUC:  {auc:.4f} ({auc_delta:+.4f})")
    print(f"  Status:   {status}")

print("\n" + "=" * 70)
print("TRAINING TIME COMPARISON")
print("=" * 70)
for model in comparison.index:
    print(f"{model}: {comparison.loc[model, 'train_time']:.2f}s")

## 10. Save Model Artifacts

In [ ]:
import joblib
import json

models_dir = MODELS_DIR
models_dir.mkdir(parents=True, exist_ok=True)

print(f"Saving models to: {models_dir}")

# Saving models
joblib.dump(rf, models_dir / 'random_forest_model.pkl')
joblib.dump(xgb, models_dir / 'xgboost_model.pkl')
joblib.dump(lgb, models_dir / 'lightgbm_model.pkl')

# Saving encoders
joblib.dump(dept_encoder, models_dir / 'label_encoder_department.pkl')
joblib.dump(salary_encoder, models_dir / 'label_encoder_salary.pkl')

# Saving feature columns
with open(models_dir / 'feature_columns.json', 'w') as f:
    json.dump(feature_cols, f)

optimal_thresholds_json = {k: float(v) for k, v in optimal_thresholds.items()}
with open(models_dir / 'optimal_threshold.json', 'w') as f:
    json.dump(optimal_thresholds_json, f, indent=2)

results_summary = comparison.to_dict('index')
for model_name in results_summary:
    results_summary[model_name] = {k: float(v) if isinstance(v, (np.integer, np.floating)) else v for k, v in results_summary[model_name].items()}

with open(models_dir / 'model_results.json', 'w') as f:
    json.dump(results_summary, f, indent=2)


for model_name, model_df in [('random_forest', rf_importance), 
                              ('xgboost', xgb_importance), 
                              ('lightgbm', lgb_importance)]:
    model_df.to_csv(models_dir / f'feature_importance_{model_name}.csv', index=False)

print("Model artifacts saved successfully!")
print(f"\nSaved files:")
print(f"  - random_forest_model.pkl")
print(f"  - xgboost_model.pkl")
print(f"  - lightgbm_model.pkl")
print(f"  - label_encoder_department.pkl")
print(f"  - label_encoder_salary.pkl")
print(f"  - feature_columns.json")
print(f"  - optimal_threshold.json")
print(f"  - model_results.json")
print(f"  - feature_importance_*.csv")

## 11. Verification: Load and Test Models

In [ ]:
def predict_attrition(employee_data, model, encoders, feature_cols, threshold=0.5):
    """
    Making prediction on new employee data.

    Args:
        employee_data (pd.DataFrame): DataFrame with employee features.
        model: Trained model.
        encoders (dict): Dict with department and salary encoders.
        feature_cols (list): List of feature column names.
        threshold (float): Classification threshold.

    Returns:
        dict: Dictionary with prediction results.
    """
    # Encoding categorical features
    data = employee_data.copy()

    # Encode salary (ordinal)
    data['salary_encoded'] = encoders['salary'].transform(data[['salary']]).flatten()

    # OneHot encode department
    dept_encoded = encoders['dept'].transform(data[['department']])
    dept_columns = [f'Dept_{cat}' for cat in encoders['dept'].categories_[0][1:]]
    for i, col in enumerate(dept_columns):
        data[col] = dept_encoded[:, i]

    # Preparing features
    X = data[feature_cols]

    # Predicting
    proba = model.predict_proba(X)[0, 1]
    prediction = int(proba >= threshold)

    return {
        'prediction': 'Leave' if prediction == 1 else 'Stay',
        'probability': proba,
        'risk_level': 'High' if proba > 0.7 else 'Medium' if proba > 0.4 else 'Low'
    }


# Testing with sample employees
encoders = {'dept': dept_encoder, 'salary': salary_encoder}
test_employees = pd.DataFrame({
    'satisfaction_level': [0.38, 0.80, 0.11],
    'last_evaluation': [0.53, 0.86, 0.88],
    'num_projects': [2, 5, 7],
    'avg_monthly_hours': [157, 262, 272],
    'years_at_company': [3, 6, 4],
    'had_work_accident': [0, 0, 0],
    'promotion_last_5_years': [0, 0, 0],
    'department': ['sales', 'sales', 'sales'],
    'salary': ['low', 'medium', 'medium']
})

print("Sample Predictions (XGBoost):")
for idx, row in test_employees.iterrows():
    result = predict_attrition(
        pd.DataFrame([row]), 
        xgb, 
        encoders, 
        feature_cols,
        threshold=optimal_thresholds['XGBoost']
    )
    print(f"Employee {idx+1}: {result['prediction']} ({result['probability']:.2%}) - {result['risk_level']} risk")

## Summary

This notebook trained and compared three advanced machine learning models for employee attrition prediction:

**Models Trained:**
- Random Forest (baseline)
- XGBoost
- LightGBM

**Key Findings:**
- All models achieved >95% accuracy
- Training times varied significantly
- Feature importance patterns were consistent across models